In [37]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import copy
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True
)

In [38]:
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Base")
model = AutoModelForImageTextToText.from_pretrained("HuggingFaceTB/SmolVLM-Base",
                                                quantization_config=quant_config,
                                                _attn_implementation="flash_attention_2" if DEVICE == "cuda" else "eager"
                                                ).to(DEVICE)


Loading weights: 100%|██████████| 657/657 [00:39<00:00, 16.54it/s] 


In [39]:
print(torch.backends.cuda.flash_sdp_enabled())

True


In [40]:
import torch.nn as nn
import bitsandbytes as bnb

In [41]:
class SwitchLayer(nn.Module):
    def __init__(self, hidden_dim, expert_network:nn.Module, n_expert:int=4, top_k:int=1):
        super().__init__()

        self.switch = nn.Linear(hidden_dim, n_expert, dtype=torch.bfloat16)
        self.noise = nn.Linear(hidden_dim, n_expert, dtype=torch.bfloat16)
        self.top_k = top_k

        self.experts = nn.ModuleList(
            [copy.deepcopy(expert_network).to(DEVICE) for _ in range(n_expert)]
        )
        self.expert_count = [0]*n_expert

        self.softplus = nn.Softplus()
        self.activation = nn.Softmax(dim=-1)
    
    def keep_top_k(self, expert_logits):
        values, indices = torch.topk(expert_logits, self.top_k, dim=-1)

        logits = torch.full_like(expert_logits, float('-inf'))
        logits.scatter_(1, indices, values)

        return logits
    
    def get_expert_count(self):
        return self.expert_count.copy()
    
    def reset_expert_count(self):
        self.expert_count = [0 for _ in self.expert_count]

    def forward(self, x):

        B, S, D = x.shape
        x_flat = x.reshape(B * S, D)

        expert_logits = self.switch(x_flat)
        noise = torch.randn_like(expert_logits)*self.softplus(self.noise(x_flat))
        expert_logits = self.activation(self.keep_top_k(expert_logits + noise))

        output = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.experts):

            idx = (expert_logits[:, i] > 0).nonzero(as_tuple=True)[0]

            if idx.numel() == 0:
                continue

            tokens = x_flat[idx]
            weights = expert_logits[idx, i].unsqueeze(-1)

            expert_out = expert(tokens)

            output[idx] += weights * expert_out

            # Track expert usage (no grad)
            self.expert_count[i] += idx.numel()

        return output.reshape(B, S, D)

In [42]:
for blocks in model.model.vision_model.encoder.layers[1::2]:
    switch = SwitchLayer(1152, blocks.mlp, 4, 2).to(DEVICE)
    blocks.mlp = switch

for blocks in model.model.text_model.layers[1::2]:
    switch = SwitchLayer(2048, blocks.mlp, 4, 2).to(DEVICE)
    blocks.mlp = switch
    

In [43]:
print(model.model.vision_model.encoder.layers)

ModuleList(
  (0): Idefics3EncoderLayer(
    (self_attn): Idefics3VisionAttention(
      (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
      (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
      (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
      (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
    )
    (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
    (mlp): Idefics3VisionMLP(
      (activation_fn): GELUTanh()
      (fc1): Linear4bit(in_features=1152, out_features=4304, bias=True)
      (fc2): Linear4bit(in_features=4304, out_features=1152, bias=True)
    )
    (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
  )
  (1): Idefics3EncoderLayer(
    (self_attn): Idefics3VisionAttention(
      (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
      (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
      (q_proj): Linear

In [44]:
print(model.model.text_model.layers)

ModuleList(
  (0): LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
      (k_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
      (v_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
      (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    )
    (mlp): LlamaMLP(
      (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
      (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
      (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
    (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  )
  (1): LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
      (k_proj): Linear4bit(in_features=2048, out_features=2048, 

In [45]:
from transformers.image_utils import load_image

In [46]:
image1 = load_image("https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg")
image2 = load_image("https://huggingface.co/spaces/merve/chameleon-7b/resolve/main/bee.jpg")

In [47]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "image"},
            {"type": "text", "text": "Can you describe the two images?"}
        ]
    },
]
# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image1, image2], return_tensors="pt")
inputs = inputs.to(DEVICE)
# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)
print(generated_texts[0])

User:<row_1_col_1><row_1_col_2><row_1_col_3><row_1_col_4>
<row_2_col_1><row_2_col_2><row_2_col_3><row_2_col_4>
<row_3_col_1><row_3_col_2><row_3_col_3><row_3_col_4>

<global-img><row_1_col_1><row_1_col_2><row_1_col_3><row_1_col_4>
<row_2_col_1><row_2_col_2><row_2_col_3><row_2_col_4>
<row_3_col_1><row_3_col_2><row_3_col_3><row_3_col_4>

<global-img>Can you describe the two images?
Assistant: <row_1_col_1> <row_1_col_2> <row_1_col_3> <row_1_col_4>
Assistant: <row_2_col_1> <row_2_col_2> <row_2_col_3> <row_2_col_4>
Assistant: <row_3_col_1> <row_3_col_2> <row_3_col_3> <row_3_col_4>
Assistant: <row_4_col_1> <row_4_col_2> <row_4_col_3> <row_4_col_4>


In [48]:
print(messages)

[{'role': 'user', 'content': [{'type': 'image'}, {'type': 'image'}, {'type': 'text', 'text': 'Can you describe the two images?'}]}]


In [49]:
print(inputs)

{'pixel_values': tensor([[[[[ 0.2392,  0.2392,  0.2392,  ...,  0.3255,  0.3255,  0.3176],
           [ 0.2392,  0.2392,  0.2392,  ...,  0.3255,  0.3255,  0.3255],
           [ 0.2392,  0.2392,  0.2392,  ...,  0.3333,  0.3333,  0.3333],
           ...,
           [ 0.4667,  0.4667,  0.4667,  ...,  0.6078,  0.6078,  0.6078],
           [ 0.4667,  0.4667,  0.4667,  ...,  0.6078,  0.6078,  0.6078],
           [ 0.4667,  0.4667,  0.4667,  ...,  0.6078,  0.6078,  0.6078]],

          [[ 0.5843,  0.5843,  0.5843,  ...,  0.6706,  0.6706,  0.6627],
           [ 0.5843,  0.5843,  0.5843,  ...,  0.6706,  0.6706,  0.6706],
           [ 0.5843,  0.5843,  0.5843,  ...,  0.6784,  0.6784,  0.6784],
           ...,
           [ 0.5059,  0.5059,  0.5059,  ...,  0.5451,  0.5451,  0.5451],
           [ 0.5059,  0.5059,  0.5059,  ...,  0.5451,  0.5451,  0.5451],
           [ 0.5059,  0.5059,  0.5059,  ...,  0.5451,  0.5451,  0.5451]],

          [[ 0.8118,  0.8118,  0.8118,  ...,  0.8824,  0.8824,  0.8745]